# Data Analysis - Lesson 13
## Pivot Tables, Crosstab & Advanced Aggregation

Complete hands-on Jupyter Notebook.

### Learning Objectives
- Use `groupby().agg()` for grouped summaries
- Build business-friendly summaries with `pd.pivot_table()`
- Use multiple dimensions and aggregation functions
- Add totals with `margins=True`
- Handle missing combinations with `fill_value`
- Use `pd.crosstab()` for categorical counts and percentages
- Understand `pivot()` vs `pivot_table()`
- Reshape data with `melt()`
- Apply these techniques to restaurant and manufacturing/QA analysis

**Primary dataset:** Seaborn's built-in `tips` dataset.

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

tips = sns.load_dataset("tips")
tips.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [2]:
tips.shape

(244, 7)

In [3]:
tips.columns

Index(['total_bill', 'tip', 'sex', 'smoker', 'day', 'time', 'size'], dtype='object')

In [4]:
tips.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 244 entries, 0 to 243
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   total_bill  244 non-null    float64 
 1   tip         244 non-null    float64 
 2   sex         244 non-null    category
 3   smoker      244 non-null    category
 4   day         244 non-null    category
 5   time        244 non-null    category
 6   size        244 non-null    int64   
dtypes: category(4), float64(2), int64(1)
memory usage: 7.4 KB


## 1. Quick Review — GroupBy

*groupby()* follows the split-apply-combine pattern. It is useful for questions such as average bill by day, revenue by day, or transaction count by day.

In [5]:
import warnings
warnings.filterwarnings('ignore')

In [6]:
tips.groupby("day")["total_bill"].mean()

day
Thur    17.682742
Fri     17.151579
Sat     20.441379
Sun     21.410000
Name: total_bill, dtype: float64

In [7]:
tips.groupby("day")["total_bill"].agg(
    ["count", "mean", "median", "sum", "min", "max"])

,count,mean,median,sum,min,max
day,,,,,,
Thur,62,17.682742,16.20,1096.33,7.51,43.11
Fri,19,17.151579,15.38,325.88,5.75,40.17
Sat,87,20.441379,18.24,1778.40,3.07,50.81
Sun,76,21.410000,19.63,1627.16,7.25,48.17


In [8]:
tips.groupby(["day", "time"])["total_bill"].mean()

day   time  
Thur  Lunch     17.664754
      Dinner    18.780000
Fri   Lunch     12.845714
      Dinner    19.663333
Sat   Lunch           NaN
      Dinner    20.441379
Sun   Lunch           NaN
      Dinner    21.410000
Name: total_bill, dtype: float64

## 2. What Is a Pivot Table?

A Pivot Table summarizes data using:

- **Rows** → `index`
- **Columns** → `columns`
- **Values** → `values`
- **Aggregation** → `aggfunc`

Example concept:

```text
                Lunch       Dinner
Thursday        Avg         Avg
Friday          Avg         Avg
Saturday        Avg         Avg
Sunday          Avg         Avg
```

## 3. Basic `pivot_table()`

In [9]:
pd.pivot_table(
    tips,
    values="total_bill",
    index="day",
    aggfunc="mean")

,total_bill
day,
Thur,17.682742
Fri,17.151579
Sat,20.441379
Sun,21.410000


This is conceptually similar to:

```python
tips.groupby("day")["total_bill"].mean()
```

Pivot tables become especially useful when we introduce a column dimension.

In [10]:
pd.pivot_table(
    tips,
    values="total_bill",
    index="day",
    columns="time",
    aggfunc="mean")

time,Lunch,Dinner
day,,
Thur,17.664754,18.780000
Fri,12.845714,19.663333
Sat,NaN,20.441379
Sun,NaN,21.410000


## 4. Different Aggregation Functions

In [11]:
# Average
pd.pivot_table(tips, values="total_bill", 
               index="day", columns="time", aggfunc="mean")

time,Lunch,Dinner
day,,
Thur,17.664754,18.780000
Fri,12.845714,19.663333
Sat,NaN,20.441379
Sun,NaN,21.410000


In [12]:
# Total
pd.pivot_table(tips, values="total_bill", 
    index="day", columns="time", aggfunc="sum")

time,Lunch,Dinner
day,,
Thur,1077.55,18.78
Fri,89.92,235.96
Sat,0.00,1778.40
Sun,0.00,1627.16


In [13]:
# Count
pd.pivot_table(tips, values="total_bill", 
    index="day", columns="time", aggfunc="count")

time,Lunch,Dinner
day,,
Thur,61,1
Fri,7,12
Sat,0,87
Sun,0,76


In [14]:
# Maximum
pd.pivot_table(
    tips, 
    values="total_bill", 
    index="day", 
    columns="time", 
    aggfunc="max")

time,Lunch,Dinner
day,,
Thur,43.11,18.78
Fri,16.27,40.17
Sat,NaN,50.81
Sun,NaN,48.17


In [15]:
# Median
pd.pivot_table(
    tips, 
    values="total_bill", 
    index="day", 
    columns="time", 
    aggfunc="median")

time,Lunch,Dinner
day,,
Thur,16.00,18.780
Fri,13.42,18.665
Sat,NaN,18.240
Sun,NaN,19.630


## 5. Multiple Aggregations

In [16]:
pd.pivot_table(
    tips,
    values="total_bill",
    index="day",
    columns="time",
    aggfunc=["mean", "max", "min"])

mean               max          min       
time      Lunch     Dinner  Lunch Dinner Lunch Dinner
day                                                  
Thur  17.664754  18.780000  43.11  18.78  7.51  18.78
Fri   12.845714  19.663333  16.27  40.17  8.58   5.75
Sat         NaN  20.441379    NaN  50.81   NaN   3.07
Sun         NaN  21.410000    NaN  48.17   NaN   7.25

Multiple aggregation functions create hierarchical columns, known as a MultiIndex.

## 6. Multiple Values

In [17]:
pd.pivot_table(
    tips,
    values=["total_bill", "tip"],
    index="day",
    columns="time",
    aggfunc="mean")

tip           total_bill           
time     Lunch    Dinner      Lunch     Dinner
day                                           
Thur  2.767705  3.000000  17.664754  18.780000
Fri   2.382857  2.940000  12.845714  19.663333
Sat        NaN  2.993103        NaN  20.441379
Sun        NaN  3.255132        NaN  21.410000

## 7. Different Aggregations for Different Values

In [18]:
pd.pivot_table(
    tips,
    values=["total_bill", "tip"],
    index="day",
    columns="time",
    aggfunc={
        "total_bill": ["mean", "sum"],
        "tip": ["mean", "max"]})

tip                            total_bill                             
       max             mean                 mean                 sum         
time Lunch Dinner     Lunch    Dinner      Lunch     Dinner    Lunch   Dinner
day                                                                          
Thur  6.70   3.00  2.767705  3.000000  17.664754  18.780000  1077.55    18.78
Fri   3.48   4.73  2.382857  2.940000  12.845714  19.663333    89.92   235.96
Sat    NaN  10.00       NaN  2.993103        NaN  20.441379     0.00  1778.40
Sun    NaN   6.50       NaN  3.255132        NaN  21.410000     0.00  1627.16

## 8. Filling Missing Combinations

`fill_value=0` fills missing cells in the resulting pivot table. It does not mean that the original data contained a real zero.

In [19]:
pd.pivot_table(
    tips,
    values="total_bill",
    index="day",
    columns="time",
    aggfunc="sum",
    fill_value=0)

time,Lunch,Dinner
day,,
Thur,1077.55,18.78
Fri,89.92,235.96
Sat,0.00,1778.40
Sun,0.00,1627.16


## 9. Grand Totals with `margins=True`

In [20]:
pd.pivot_table(
    tips,
    values="total_bill",
    index="day",
    columns="time",
    aggfunc="sum",
    margins=True)

time,Lunch,Dinner,All
day,,,
Thur,1077.55,18.78,1096.33
Fri,89.92,235.96,325.88
Sat,0.00,1778.40,1778.40
Sun,0.00,1627.16,1627.16
All,1167.47,3660.30,4827.77


In [21]:
pd.pivot_table(
    tips,
    values="total_bill",
    index="day",
    columns="time",
    aggfunc="sum",
    margins=True,
    margins_name="Grand Total")

time,Lunch,Dinner,Grand Total
day,,,
Thur,1077.55,18.78,1096.33
Fri,89.92,235.96,325.88
Sat,0.00,1778.40,1778.40
Sun,0.00,1627.16,1627.16
Grand Total,1167.47,3660.30,4827.77


## 10. Business Analysis — Restaurant Revenue

In [22]:
daily_revenue = pd.pivot_table(
    tips, 
    values="total_bill", 
    index="day", 
    aggfunc="sum")

daily_revenue

,total_bill
day,
Thur,1096.33
Fri,325.88
Sat,1778.40
Sun,1627.16


In [23]:
daily_revenue.sort_values("total_bill", ascending=False)

,total_bill
day,
Sat,1778.40
Sun,1627.16
Thur,1096.33
Fri,325.88


In [24]:
revenue_matrix = pd.pivot_table(
    tips,
    values="total_bill",
    index="day",
    columns="time",
    aggfunc="sum",
    fill_value=0)

revenue_matrix

time,Lunch,Dinner
day,,
Thur,1077.55,18.78
Fri,89.92,235.96
Sat,0.00,1778.40
Sun,0.00,1627.16


In [25]:
average_bill_matrix = pd.pivot_table(
    tips,
    values="total_bill",
    index="day",
    columns="time",
    aggfunc="mean")

average_bill_matrix

time,Lunch,Dinner
day,,
Thur,17.664754,18.780000
Fri,12.845714,19.663333
Sat,NaN,20.441379
Sun,NaN,21.410000


In [26]:
transaction_matrix = pd.pivot_table(
    tips,
    values="total_bill",
    index="day",
    columns="time",
    aggfunc="count",
    fill_value=0)

transaction_matrix

time,Lunch,Dinner
day,,
Thur,61,1
Fri,7,12
Sat,0,87
Sun,0,76


## 11. GroupBy vs Pivot Table

**GroupBy** is generally better for analytical calculations and transformation pipelines.

**Pivot Table** is generally better for business summaries, matrix-style analysis, and report-oriented output.

Mental model:

**GroupBy = analytical calculation**

**Pivot Table = analytical calculation + report-friendly layout**

## 12. `pivot()` vs `pivot_table()`

In [27]:
df = pd.DataFrame({
    "Date": ["2026-01-01", "2026-01-02", "2026-01-03"],
    "Metric": ["Sales", "Sales", "Sales"],
    "Value": [100, 120, 150]
})

df.pivot(index="Date", columns="Metric", values="Value")

Metric,Sales
Date,
2026-01-01,100
2026-01-02,120
2026-01-03,150


In [28]:
duplicate_df = pd.DataFrame({
    "Day": ["Mon", "Mon", "Tue"],
    "Shift": ["A", "A", "B"],
    "Sales": [100, 120, 150]
})
duplicate_df

,Day,Shift,Sales
0,Mon,A,100
1,Mon,A,120
2,Tue,B,150


`pivot()` cannot resolve duplicate index-column combinations because it does not know which aggregation to use.

In [29]:
pd.pivot_table(
    duplicate_df,
    values="Sales",
    index="Day",
    columns="Shift",
    aggfunc="sum"
)

Shift,A,B
Day,,
Mon,220.0,NaN
Tue,NaN,150.0


## 13. Crosstab

`pd.crosstab()` is especially convenient for categorical frequency analysis.

In [30]:
pd.crosstab(tips["day"], tips["smoker"])

smoker,Yes,No
day,,
Thur,17,45
Fri,15,4
Sat,42,45
Sun,19,57


In [31]:
pd.crosstab(tips["sex"], tips["smoker"])

smoker,Yes,No
sex,,
Male,60,97
Female,33,54


In [32]:
pd.crosstab(tips["sex"], tips["smoker"], margins=True)

smoker,Yes,No,All
sex,,,
Male,60,97,157
Female,33,54,87
All,93,151,244


## 14. Crosstab Percentages

In [33]:
# Percentage within each row
pd.crosstab(
    tips["sex"],
    tips["smoker"],
    normalize="index"
) * 100

smoker,Yes,No
sex,,
Male,38.216561,61.783439
Female,37.931034,62.068966


In [34]:
row_percentages = pd.crosstab(
    tips["sex"], tips["smoker"], normalize="index"
) * 100
row_percentages.sum(axis=1)

sex
Male      100.0
Female    100.0
dtype: float64

In [35]:
# Percentage within each column
pd.crosstab(
    tips["sex"],
    tips["smoker"],
    normalize="columns"
) * 100

smoker,Yes,No
sex,,
Male,64.516129,64.238411
Female,35.483871,35.761589


In [36]:
# Percentage of the entire dataset
pd.crosstab(
    tips["sex"],
    tips["smoker"],
    normalize="all"
) * 100

smoker,Yes,No
sex,,
Male,24.590164,39.754098
Female,13.524590,22.131148


## 15. Crosstab with Numerical Values

In [37]:
pd.crosstab(
    tips["sex"],
    tips["smoker"],
    values=tips["total_bill"],
    aggfunc="mean"
)

smoker,Yes,No
sex,,
Male,22.284500,19.791237
Female,17.977879,18.105185


In [38]:
pd.crosstab(
    tips["day"],
    tips["time"],
    values=tips["total_bill"],
    aggfunc="sum"
)

time,Lunch,Dinner
day,,
Thur,1077.55,18.78
Fri,89.92,235.96
Sat,0.00,1778.40
Sun,0.00,1627.16


## 16. Pivot Table vs Crosstab

| Feature | GroupBy | Pivot Table | Crosstab |
|---|---|---|---|
| Main purpose | Aggregation | Summary matrix | Frequency/categorical analysis |
| Numerical aggregation | Excellent | Excellent | Possible |
| Categorical counts | Good | Good | Excellent |
| Matrix layout | Limited | Excellent | Excellent |
| Percentages | Manual | Manual | Very convenient |
| Reporting | Good | Excellent | Excellent |

Rule of thumb:
- Numerical aggregation → Pivot Table
- Categorical frequency/percentage → Crosstab

## 17. Manufacturing / QA Example

In [39]:
quality = pd.DataFrame({
    "Line": ["A", "A", "A", "A", "B", "B", "B", "B",
             "A", "A", "B", "B"],
    "Shift": ["1", "1", "2", "2", "1", "1", "2", "2",
              "1", "2", "1", "2"],
    "Defect_Type": [
        "Scratch", "Dent", "Scratch", "Dimension",
        "Dent", "Scratch", "Dimension", "Scratch",
        "Scratch", "Dent", "Dent", "Dimension"
    ],
    "Measurement": [
        10.02, 10.05, 9.98, 10.10,
        10.01, 9.97, 10.08, 10.12,
        10.03, 10.06, 10.00, 10.09
    ],
    "Defect": [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
})
quality

,Line,Shift,Defect_Type,Measurement,Defect
0,A,1,Scratch,10.02,1
1,A,1,Dent,10.05,1
2,A,2,Scratch,9.98,1
3,A,2,Dimension,10.10,1
4,B,1,Dent,10.01,1
5,B,1,Scratch,9.97,1
6,B,2,Dimension,10.08,1
7,B,2,Scratch,10.12,1
8,A,1,Scratch,10.03,1
9,A,2,Dent,10.06,1


In [40]:
# Defect count by Line and Shift
pd.pivot_table(
    quality,
    values="Defect",
    index="Line",
    columns="Shift",
    aggfunc="sum",
    fill_value=0
)

Shift,1,2
Line,,
A,3,3
B,3,3


In [41]:
# Average measurement by Line and Shift
pd.pivot_table(
    quality,
    values="Measurement",
    index="Line",
    columns="Shift",
    aggfunc="mean"
)

Shift,1,2
Line,,
A,10.033333,10.046667
B,9.993333,10.096667


In [42]:
# Defect type by Line
pd.crosstab(quality["Line"], quality["Defect_Type"])

Defect_Type,Dent,Dimension,Scratch
Line,,,
A,2,1,3
B,2,2,2


In [43]:
# Defect type by Shift
pd.crosstab(quality["Shift"], quality["Defect_Type"])

Defect_Type,Dent,Dimension,Scratch
Shift,,,
1,3,0,3
2,1,3,2


In [44]:
# Defect type percentage within each Line
pd.crosstab(
    quality["Line"],
    quality["Defect_Type"],
    normalize="index"
) * 100

Defect_Type,Dent,Dimension,Scratch
Line,,,
A,33.333333,16.666667,50.000000
B,33.333333,33.333333,33.333333


## 18. Defect Count vs Defect Rate

Raw defect counts do not always provide a fair comparison.

A common KPI is:

**Defect Rate = Defects / Produced × 100**

Always consider the appropriate denominator when comparing production groups.

In [45]:
quality["Produced"] = [
    1000, 1000, 900, 900,
    800, 800, 750, 750,
    950, 850, 780, 720
]

quality.head()

,Line,Shift,Defect_Type,Measurement,Defect,Produced
0,A,1,Scratch,10.02,1,1000
1,A,1,Dent,10.05,1,1000
2,A,2,Scratch,9.98,1,900
3,A,2,Dimension,10.10,1,900
4,B,1,Dent,10.01,1,800


In [46]:
# Total production by Line and Shift
pd.pivot_table(
    quality,
    values="Produced",
    index="Line",
    columns="Shift",
    aggfunc="sum",
    fill_value=0)

Shift,1,2
Line,,
A,2950,2650
B,2380,2220


In [47]:
# Total defects by Line and Shift
pd.pivot_table(
    quality,
    values="Defect",
    index="Line",
    columns="Shift",
    aggfunc="sum",
    fill_value=0)

Shift,1,2
Line,,
A,3,3
B,3,3


## 19. Reshaping with `melt()`

`melt()` converts wide data into long data.

Wide:

```text
Day     Lunch    Dinner
Thur     500      700
Fri      400      600
Sat      800     1200
```

Long:

```text
Day     Time      Revenue
Thur    Lunch      500
Thur    Dinner     700
Fri     Lunch      400
Fri     Dinner     600
Sat     Lunch      800
Sat     Dinner    1200
```

In [48]:
sales = pd.DataFrame({
    "Day": ["Thur", "Fri", "Sat"],
    "Lunch": [500, 400, 800],
    "Dinner": [700, 600, 1200]
})

sales

,Day,Lunch,Dinner
0,Thur,500,700
1,Fri,400,600
2,Sat,800,1200


In [49]:
sales_long = sales.melt(
    id_vars="Day",
    var_name="Time",
    value_name="Revenue")

sales_long

,Day,Time,Revenue
0,Thur,Lunch,500
1,Fri,Lunch,400
2,Sat,Lunch,800
3,Thur,Dinner,700
4,Fri,Dinner,600
5,Sat,Dinner,1200


## 20. Mini Project — Restaurant Performance Report

Create:
1. Day × Time total revenue
2. Day × Time average bill
3. Day × Time transaction count
4. Day × Smoker count
5. Day × Smoker row percentages
6. Gender × Smoker average bill

In [50]:
# Solution reference — Table 1
revenue = pd.pivot_table(
    tips, values="total_bill", index="day",
    columns="time", aggfunc="sum", fill_value=0)

revenue

time,Lunch,Dinner
day,,
Thur,1077.55,18.78
Fri,89.92,235.96
Sat,0.00,1778.40
Sun,0.00,1627.16


In [51]:
# Solution reference — Table 2
avg_bill = pd.pivot_table(
    tips, 
    values="total_bill", 
    index="day",
    columns="time", 
    aggfunc="mean")

avg_bill

time,Lunch,Dinner
day,,
Thur,17.664754,18.780000
Fri,12.845714,19.663333
Sat,NaN,20.441379
Sun,NaN,21.410000


In [52]:
# Solution reference — Table 3
transactions = pd.pivot_table(
    tips, 
    values="total_bill", 
    index="day",
    columns="time", 
    aggfunc="count", 
    fill_value=0)

transactions

time,Lunch,Dinner
day,,
Thur,61,1
Fri,7,12
Sat,0,87
Sun,0,76


In [53]:
# Solution reference — Table 4
customer_mix = pd.crosstab(tips["day"], tips["smoker"])
customer_mix

smoker,Yes,No
day,,
Thur,17,45
Fri,15,4
Sat,42,45
Sun,19,57


In [54]:
# Solution reference — Table 5
customer_mix_pct = pd.crosstab(
    tips["day"], tips["smoker"], 
    normalize="index") * 100

customer_mix_pct

smoker,Yes,No
day,,
Thur,27.419355,72.580645
Fri,78.947368,21.052632
Sat,48.275862,51.724138
Sun,25.000000,75.000000


In [55]:
# Solution reference — Table 6
gender_smoker_avg = pd.crosstab(
    tips["sex"], tips["smoker"],
    values=tips["total_bill"], aggfunc="mean")

gender_smoker_avg

smoker,Yes,No
sex,,
Male,22.284500,19.791237
Female,17.977879,18.105185


## 21. Advanced Challenge

In [56]:
advanced_report = pd.pivot_table(
    tips,
    values="total_bill",
    index="day",
    columns="time",
    aggfunc=["count", "mean", "median", "sum", "max"])

advanced_report

count              mean            median              sum           \
time Lunch Dinner      Lunch     Dinner  Lunch  Dinner    Lunch   Dinner   
day                                                                        
Thur    61      1  17.664754  18.780000  16.00  18.780  1077.55    18.78   
Fri      7     12  12.845714  19.663333  13.42  18.665    89.92   235.96   
Sat      0     87        NaN  20.441379    NaN  18.240     0.00  1778.40   
Sun      0     76        NaN  21.410000    NaN  19.630     0.00  1627.16   

        max         
time  Lunch Dinner  
day                 
Thur  43.11  18.78  
Fri   16.27  40.17  
Sat     NaN  50.81  
Sun     NaN  48.17

## 22. Interview Questions

1. What is a Pivot Table?
2. What is the difference between `pivot()` and `pivot_table()`?
3. What is the difference between `groupby()` and `pivot_table()`?
4. What does `margins=True` do?
5. What does `fill_value=0` do?
6. What is `pd.crosstab()` mainly used for?
7. What does `normalize="index"` mean?
8. What does `normalize="columns"` mean?
9. Why can percentages be more useful than counts?
10. What is a MultiIndex?

### Short Answers
- Pivot Table: summarized data arranged by row/column dimensions with aggregation.
- `pivot()`: reshape only; duplicate combinations cause an error.
- `pivot_table()`: reshape + aggregation.
- `margins=True`: adds totals.
- `fill_value`: fills missing output cells.
- Crosstab: categorical frequency/contingency analysis.
- `normalize="index"`: percentages across rows.
- `normalize="columns"`: percentages down columns.
- Percentages account for group size.
- MultiIndex: hierarchical labels with multiple levels.

## 23. Assignment

### Part A — Pivot Tables
1. Average `total_bill` by `day` and `time`.
2. Total `total_bill` by `day` and `time`.
3. Transaction count by `day` and `time`.
4. Mean, median, sum, and max for `total_bill` by `day` and `time`.
5. Add grand totals.

### Part B — Crosstab
6. Count table of `day × smoker`.
7. Smoker percentages within each day.
8. `sex × smoker` row percentages.
9. Average `total_bill` by `sex × smoker`.

### Part C — Manufacturing / QA
10. Line × Shift defect-count matrix.
11. Line × Shift average-measurement matrix.
12. Line × Defect_Type frequency table.
13. Defect-type percentages within each Line.
14. Explain why defect count alone may be insufficient when comparing production lines.

## 24. Final Challenge

Answer this business question programmatically:

> For each production line and shift, show total production, total defects, and defect rate in a single analysis.

Think about:

```text
Line
  ↓
Shift
  ↓
Production
Defects
Defect Rate
```

### Key Takeaway

A strong analyst does not create a Pivot Table simply because they know the syntax.

First ask:

**What business question am I trying to answer?**

Then choose the appropriate tool:

`GroupBy` → analytical grouping  
`Pivot Table` → business summary matrix  
`Crosstab` → categorical frequency/percentage  
`melt()` → wide → long reshaping

The analytical question comes first; the tool comes second.

In [57]:
# Your practice area
# Try solving the Final Challenge here without looking at the examples.

# summary = ...
# summary["Defect_Rate"] = ...
# summary